In [ ]:
import numpy as np
from qubicpack.qubicfp import qubicfp
import sys,os
import glob

import matplotlib.pyplot as plt
import matplotlib.mlab as mlab

In [ ]:
day = '2023-05-27'
data_dir = '/home/qubic/Calib-TD/'+day+'/'
words = ['DomeOpened']
keywords = ['*{}*'.format(word) for word in words]
for keyword in keywords:
    dirs = np.sort(glob.glob(data_dir+keyword))
    print(dirs)

In [ ]:
data_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/data/Flux_jumps/"
soft_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/lib/Calibration/"
dict_path = "/home/belen/Doctorado/qubic-dev/qubic/qubic/dicts/"
sys.path.append(os.path.abspath(soft_path))

In [ ]:
import Qfluxjumps as jr
import Qsaturation as qsat

## 04.48.05 skydip 2.25V

In [ ]:
dataset0 = dirs[0]
a = qubicfp()
a.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation(sat_value=4.19e6, TES_number=256) 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2023.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
thr = 2e5#, 3e5, 5e5]  # Multiple thresholds for better detection
window_size = 300  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}


In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
jr.save_results(results, output_dir=data_path + "year_23_day_2705_04.48.05", dataset_name="23_2705_04.48.05")

In [ ]:
dirs[1]

## skyscan 12.40.21 2.25V

In [ ]:
dataset0 = dirs[1]
a1 = qubicfp()
a1.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a1.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2023.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
iv_idx = np.where(iv_mask == False)[0]
iv_tod = todarray[iv_idx, :]
iv_idx

In [ ]:
thr = 2e5#, 3e5, 5e5]  # Multiple thresholds for better detection
window_size = 500  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}


In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
len(TES_yes)

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
import time 

In [ ]:
t0 = time.time()

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
t1 = time.time()

In [ ]:
t1-t0

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
for i in TES_with_dt_jumps:
    print("TES:", i, np.mean(np.abs(metrics_haar[i]['z_scores']))/np.mean(np.abs(metrics_dt[i]['z_scores'])))

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
jr.save_results(results, output_dir=data_path + "year_23_day_2705_12.40.21", dataset_name="23_2705_12.40.21")

In [ ]:
dirs[2]

## 16.28.40 skyscan 2.5V

In [ ]:
dataset0 = dirs[2]
a2 = qubicfp()
a2.read_qubicstudio_dataset(dataset0)

In [ ]:
tod = a2.tod()
timeaxis = tod[0]
todarray = tod[1]
init = timeaxis[0]
tt = timeaxis - init

print('number of timesamples along every TES in this dataset:', np.shape(todarray[0,:])) 

In [ ]:
tt[-1]/60/60

In [ ]:
# TES_yes: idx TES with flux jumps found by haar filter
# TES_no: idx TES with no flux jumps found
# TES_yes_dt: refinament of idx TES with flux jumps using DT (could be equal to TES_yes)
# jump_data: "idx", xc", "xcf", "nc"
# dt_jump_data: "idx", "xcdt", "xcfdt", "ncdt"
# corrected data no dt: idx and TES corrected without DT 
# offset no dt: amplitudes of jumps without DT
# corrected data dt: idx and TES corrected with DT 
# offset dt: amplitudes of jumps with DT
# metrics no dt: metrics without DT 
# metrics dt: metrics with DT

results = {
    'TES_yes': [],
    'TES_no': [],
    'TES_yes_dt': [],
    'jump_data': {},
    'dt_jump_data': {},
    'corrected_data_nodt': {},
    'offset_nodt': {},
    'corrected_data_dt': {},
    'offset_dt': {},
    'metrics_nodt': {}, 
    'metrics_dt':{}
    }

In [ ]:
sat = qsat.saturation() 
sat_mask, sat_idx, sat_frac, n_sat = sat.detect_saturation(todarray)

#IV class. You have to load the directory and the filename of the IV analysis
#IVdict2025.yaml or IVdict2023.yaml are the dicts with the IV information from 2025 and 2022-2023
iv = jr.badIV(directory=dict_path, n_times=10, TES_number=256, filename="IVdict2023.yaml") 
iv_mask = iv.select_badIV()

good_mask = sat_mask & iv_mask        # Create a mask with both saturation and bad IV
good_idx  = np.where(good_mask)[0]    # index of TES with good IV and no saturation
good_tod  = todarray[good_mask, :]    # TOD of good TES

print(good_idx)

In [ ]:
thr = 2.5e5#, 3e5, 5e5]  # Multiple thresholds for better detection
window_size = 300  # Optimized for this dataset length
fluxjumps = jr.fluxjumps(thr=thr, window_size=window_size)

In [ ]:
jump_data = {}
for idx_good in good_idx:
    nc, xc, xcf = fluxjumps.jumps_detection(
        todarray[idx_good], 
        consec=True,   # Merge consecutive jumps
        nc_cond=True  # Re-threshold if too many jumps
    )
    jump_data[idx_good] = {'nc': nc, 'xc': xc, 'xcf': xcf}

In [ ]:
# Classify TES
TES_yes = [idx for idx in good_idx if jump_data[idx]['nc'] > 0]
TES_no = [idx for idx in good_idx if jump_data[idx]['nc'] == 0]

In [ ]:
results['TES_yes'] = TES_yes
results['TES_no'] = TES_no
results['jump_data'] = jump_data

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=False)

In [ ]:
# Step 2: Decision Tree Refinement
dt = jr.DT(thr_count=1e3, thr_amp=2.5e5, tol=1e2, depth=True)
dt_jump_data = {}
TES_with_dt_jumps = []
for idx in TES_yes:
    nc = jump_data[idx]['nc']
    xcdt, xcfdt = dt.calculate_levels(tt, todarray[idx], nc, consec=True)
    dt_jump_data[idx] = {'xcdt': xcdt, 'xcfdt': xcfdt, 'ncdt': len(xcdt)}
    if len(xcdt) > 0:
        TES_with_dt_jumps.append(idx)

In [ ]:
results['dt_jump_data'] = dt_jump_data
results['TES_yes_dt'] = TES_with_dt_jumps

In [ ]:
%matplotlib inline

In [ ]:
jr.plot_jump_detections(tt, todarray, results, DT=True)

In [ ]:
corr = jr.correction(region_off=5, region_amp=3, change_mode="const")
corrected_tod = {}
offset = {}
for idx in TES_yes:
    if idx in dt_jump_data:
        xcdt = dt_jump_data[idx]['xcdt']
        xcfdt = dt_jump_data[idx]['xcfdt']
        nc_dt = dt_jump_data[idx]['ncdt']
        if nc_dt > 0:
            offset[idx] = corr.calculate_amplitude(todarray[idx], xcdt, xcfdt, nc_dt)
            corrected_tod[idx] = corr.correct_TOD(todarray[idx], offset[idx], xcdt, xcfdt, nc_dt)

In [ ]:
corrected_tod_nodt = {}
offset_nodt = {}
for idx in TES_yes:
    if idx in jump_data:
        xc = jump_data[idx]['xc']
        xcf = jump_data[idx]['xcf']
        nc = jump_data[idx]['nc']
        if nc > 0:
            offset_nodt[idx] = corr.calculate_amplitude(todarray[idx], xc, xcf, nc)
            corrected_tod_nodt[idx] = corr.correct_TOD(todarray[idx], offset_nodt[idx], xc, xcf, nc)

In [ ]:
results['corrected_data_dt'] = corrected_tod
results['offset_dt'] = offset

results['corrected_data_nodt'] = corrected_tod_nodt
results['offset_nodt'] = offset_nodt

In [ ]:
metrics_dt, global_rms_dt = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=True, use_corrected=True
)

# without DT
metrics_haar, global_rms_haar = jr.compute_residual_metrics_from_results(
    results, todarray, window=4000, robust=False, use_dt=False, use_corrected=True
)

In [ ]:
results['metrics_nodt'] = metrics_haar
results['metrics_dt'] = metrics_dt

In [ ]:
jr.plot_corrections(tt, todarray, results)

In [ ]:
jr.save_results(results, output_dir=data_path + "year_23_day_2705_16.28.40", dataset_name="23_2705_16.28.40")